In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

# Automatically determine the project root (the folder containing "main")
root = Path.cwd().parent    # current dir = main/second → parent = main
sys.path.append(str(root))

In [4]:
import os
os.environ['DATABASE_URL'] = 'sqlite:///resumes.db'
import torch
import pandas as pd
from datasets import load_from_disk
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch import nn
import numpy as np
from transformers import RobertaTokenizer, RobertaModel, BatchEncoding

from embeddings.contrastive_learning import AUGMENTATION_FNS, ContrastiveLearningModel, ContrastiveLearningDataset, load_model
from infrastructure.database import init_db, import_postings_from_csv, import_resumes_from_csv
from embeddings.embed_stage2 import (
    fetch_all_postings_text, 
    fetch_all_resumes_text, 
    doc_sim_score
)

In [5]:
TOKENIZER = RobertaTokenizer.from_pretrained('roberta-base')
DEVICE = ('cuda:0' if torch.cuda.is_available() else 'cpu')

In [6]:
posting_db_url = 'sqlite:///postings.db'
resume_db_url = 'sqlite:///resumes.db'

In [7]:
# fetch_all_resumes_text(resume_db_url)

In [8]:
# init_db
# import_postings_from_csv()

In [9]:
# init_db()
# import_resumes_from_csv(csv_path='resume_data/sample_resumes.csv')

In [10]:
'''
How to get a single document similarity score from a list of word embeddings from both job and resume?

- Unweighted average
- K-means cluster --> concatenate cluster centers
- Soft k-means cluster
- Discrete cosine transform

'''

'\nHow to get a single document similarity score from a list of word embeddings from both job and resume?\n\n- Unweighted average\n- K-means cluster --> concatenate cluster centers\n- Soft k-means cluster\n- Discrete cosine transform\n\n'

In [11]:
postings = fetch_all_postings_text(posting_db_url)
# resumes = fetch_all_resumes_text(resume_db_url)
df_resumes = pd.read_csv('resume_data/Resume.csv')

In [12]:
resumes = df_resumes['Resume_str'].tolist()

In [13]:
# postings[4]

In [14]:
# resumes[4]

In [21]:
# (11,4), (2,4)
# bert_model = RobertaModel.from_pretrained('./roberta-tuned-v1', add_pooling_layer=False, output_hidden_states=True)
# bert_model = RobertaModel.from_pretrained('roberta-base', add_pooling_layer=False, output_hidden_states=True)
# cl_model = ContrastiveLearningModel(bert_model, out_embed_dim=588).to('cuda:0')
# cl_model = load_model(cl_model, './temp/contrastive_learning.pth')

scores = []

for i in (list(range(len(postings)))):
#     if i != 48:
#         continue
    for j in tqdm(list(range(len(resumes)))):
        score = doc_sim_score(postings[0], resumes[j], dist_func='soft_align')
        scores.append((i, j, score))
#         break
    break

#     break
# p = embed_text(resumes[8], 'concat_last_four')
# r = embed_text(resumes[9], 'concat_last_four')
# doc_sim_score_with_cl(p, r, comp_type='wmd')

100%|██████████| 2484/2484 [00:45<00:00, 54.31it/s]


In [22]:
scores.sort(key=lambda x: x[2], reverse=True)

In [23]:
# [s for s in scores if s[1] == 8]
scores

[(0, 208, 13.173136234283447),
 (0, 145, 12.548057556152344),
 (0, 197, 12.11866807937622),
 (0, 313, 11.367962837219238),
 (0, 346, 10.998374462127686),
 (0, 661, 10.813796043395996),
 (0, 168, 10.804396629333496),
 (0, 1927, 10.612525463104248),
 (0, 201, 10.590498447418213),
 (0, 1311, 10.55198049545288),
 (0, 804, 10.526955604553223),
 (0, 645, 10.444314002990723),
 (0, 764, 10.261810779571533),
 (0, 2313, 10.192076206207275),
 (0, 2130, 10.161909580230713),
 (0, 115, 10.087296962738037),
 (0, 435, 9.994041919708252),
 (0, 2121, 9.817569732666016),
 (0, 1411, 9.749947547912598),
 (0, 1206, 9.699198722839355),
 (0, 2304, 9.662180423736572),
 (0, 165, 9.613186359405518),
 (0, 206, 9.6093111038208),
 (0, 2361, 9.589176177978516),
 (0, 2094, 9.565062046051025),
 (0, 581, 9.534933090209961),
 (0, 7, 9.457011699676514),
 (0, 1242, 9.455002784729004),
 (0, 1015, 9.388269424438477),
 (0, 205, 9.331235885620117),
 (0, 646, 9.325166702270508),
 (0, 2071, 9.32320261001587),
 (0, 193, 9.309635

In [24]:
postings[0]

'Job descriptionA leading real estate firm in New Jersey is seeking an administrative Marketing Coordinator with some experience in graphic design. You will be working closely with our fun, kind, ambitious members of the sales team and our dynamic executive team on a daily basis. This is an opportunity to be part of a fast-growing, highly respected real estate brokerage with a reputation for exceptional marketing and extraordinary culture of cooperation and inclusion.Who you are:You must be a well-organized, creative, proactive, positive, and most importantly, kind-hearted person. Please, be responsible, respectful, and cool-under-pressure. Please, be proficient in Adobe Creative Cloud (Indesign, Illustrator, Photoshop) and Microsoft Office Suite. Above all, have fantastic taste and be a good-hearted, fun-loving person who loves working with people and is eager to learn.Role:Our office is a fast-paced environment. You’ll work directly with a Marketing team and communicate daily with ot

In [34]:
resumes[201]

'         MARKETING SPECIALIST GRAPHIC DESIGNER           Professional Summary    Accomplished, creative marketing professional with proven success in graphic design, corporate marketing communications, project and event management and print production management. Recognized for the ability to perform multiple tasks at one time while meeting multiple client needs, completing projects efficiently and within or under budget, and having a high attention to detail. Respected for always setting and meeting high quality standards, being a team player willing to do whatever is needed to get the job done, and building and maintaining honest and loyal relationships. Seeking creative position that will allow me to use my creative abilities and business acumen to bring a brand to life.       Core Qualifications          Adobe  Photoshop, Illustrator and InDesign as well as Microsoft Office programs Powerpoint, Word and Excel. Areas of knowledge and expertise include:  Art Direction (design, illus

In [ ]:
###### torch.set_float32_matmul_precision('highest')
ll = nn.TripletMarginLoss(margin=2, reduction='mean')
x = torch.randn(16, 20, 128)
y = torch.randn(16, 59, 128)
z = torch.randn(16, 78, 128)
# ll(x,y,z)

In [152]:
# torch.matmul(x, y.transpose(1,2))
mask = torch.ones(16, 20, 128, dtype=bool)
mask[:]
X = torch.randn(16, 20, 128)
X

tensor([[[-0.5074,  0.6456,  0.0627,  ...,  1.4339,  1.4598,  0.6911],
         [ 0.6769, -0.6305,  1.4965,  ...,  0.3071, -0.5117, -0.2067],
         [-1.4282, -0.4100,  0.4808,  ..., -0.5921, -1.0037,  1.1876],
         ...,
         [-0.0815,  0.9946,  0.6107,  ...,  0.4782, -1.3927,  0.3651],
         [-0.9729,  1.3114,  0.6697,  ...,  1.2307,  0.7235, -0.6874],
         [-0.5364, -1.0250, -0.2753,  ..., -0.2468, -0.3026,  2.1389]],

        [[ 0.2693, -1.6578,  0.2199,  ...,  0.5011,  1.1775,  0.9088],
         [ 1.6300,  0.5477, -0.6694,  ..., -0.2469, -0.0966, -2.0516],
         [ 0.2596,  1.3851, -0.8337,  ...,  0.3786,  0.0634,  0.1645],
         ...,
         [ 1.0545, -2.9513, -1.6314,  ...,  0.9906,  0.9881,  1.0028],
         [-0.1108,  0.7447, -1.4254,  ...,  0.2376, -1.1245,  0.3419],
         [ 0.1021,  0.9599,  1.0310,  ...,  2.1958,  0.7041, -1.6799]],

        [[-1.6297,  0.1698, -0.5317,  ..., -1.2757, -0.7109,  1.3798],
         [ 0.6343, -0.9593, -1.4659,  ..., -0

In [153]:
lm_dataset = load_from_disk('./temp/lm_dataset')
dataset = ContrastiveLearningDataset(lm_dataset, 'train', TOKENIZER, AUGMENTATION_FNS, num_sample=5000)


PicklingError: Can't pickle <function sentence_dropout at 0x15523127f130>: it's not the same object as contrastive_learning.sentence_dropout

In [ ]:
loader = DataLoader(dataset, batch_size=16, shuffle=True)

In [ ]:
split_triplet_data = lambda x: (BatchEncoding({k: v[:, i] for k,v in x.items()}) for i in range(3))
anchor, pos, neg = split_triplet_data(next(iter(loader)))

In [ ]:
anchor['attention_mask']#.unsqueeze(2)#.expand(-1, -1, 384).shape

In [ ]:
# data = TOKENIZER(postings[48], truncation=True, padding='max_length', max_length=510+2, return_tensors='pt').to(DEVICE)
Z = cl_model(anchor.to(DEVICE))
Z

In [ ]:
res = torch.where(anchor['attention_mask'].to(bool).unsqueeze(2), Z, torch.nan).nanmean(dim=1)

In [ ]:
res1 = torch.empty(16, 384)
for batch in range(Z.shape[0]):
    for embed_ind in range(Z.shape[2]):
        res1[batch, embed_ind] = torch.where(anchor['attention_mask'][batch].to(bool), Z[batch, :, embed_ind], torch.nan).nanmean()
#         if anchor['attention_mask'][batch].sum


In [ ]:
torch.allclose(res.cpu(),res1)

In [ ]:
dist = lambda a,b: torch.norm(a-b, dim=2)
(dist(x,y) - dist(x,z) + 2).mean()


In [ ]:
doc_sim_score_with_cl(cl_model, postings[29], resumes[4], 'cuda:0')

In [ ]:
from datasets import load_dataset


In [ ]:
dataset = load_dataset('csv', data_files={'train': './linkedin_data/postings.csv'})

In [ ]:
dataset['train'][0]

In [ ]:

# import pandas as pd
# df = pd.read_csv('linkedin_data/postings.csv')
# df.sample(n=5000, random_state=1).to_csv('temp/postings5k.csv')